# FAISS Vector Retrieval + Hybrid Graph Notebook (Colab-safe ready)

This notebook runs retrieval after `index.faiss` and `payloads.jsonl` are available, and (when a structural graph is loadable) demonstrates the **primary hybrid pipeline**:

```text
query → embed → vector seed retrieval → graph expansion + validity/authority overlays → optional LLM generation
```

## Runtime profiles (feature 005)

| Profile | Target | Defaults |
| --- | --- | --- |
| **`colab_safe`** (default) | ~12GB hosted notebook (free Colab) | Prefer graph **pickle**; no silent JSONL rebuild; heavy exports/benchmarks **off**; conservative `TOP_K`/`TOP_N`/`HYBRID_MAX_CONTEXT` |
| **`unconstrained`** | Local / high-RAM | Fuller demos; optional JSONL rebuild; optional CSV/cache export |

Set near the config cell:

```python
RUNTIME_PROFILE = "colab_safe"   # or "unconstrained"
```

## Staged run order (Colab-safe)

```text
Stage A  config + profile + load plan / preflight
Stage B  FAISS + embedder + vector smoke query
Stage C  structural graph (pickle preferred) + optional overlays + hybrid smoke
Stage D  optional remote generation
Stage E  opt-in heavy demos only (CSV export, cache export, large benchmark, graph-guided)
```

Rules:
1. After **Stage B**, pure vector queries work **without** loading the graph.
2. Hybrid-labeled helpers work only after successful **Stage C**.
3. Default Colab-safe **Run all** does **not** execute Stage E bodies unless opt-in flags are true.
4. Hybrid is **never** silently labeled when only vector retrieval ran.

## Artifact packs

**Vector-only Colab pack:** `data/faiss_index/{index.faiss,payloads.jsonl,payload_cache.sqlite?}`

**Hybrid Colab pack:** vector pack + `data/graph/knowledge_graph.gpickle` (build locally via `scripts/build_kg_pickle.py`). Overlays optional. Full structural v2 JSONL **not** required on Colab when pickle is present.

Operator guide: [`specs/005-colab-ram-fit/quickstart.md`](../specs/005-colab-ram-fit/quickstart.md)

**Architecture**
- Store: [`SQLitePayloadFaissVectorStore`](../src/retrieval/sqlite_faiss_store.py) — FAISS + rebuild-if-stale `payload_cache.sqlite`
- Retrieval: [`VectorRetriever`](../src/retrieval/retriever.py)
- Knowledge graph: full [`src/knowledge_graph/`](../src/knowledge_graph/) package — [`KnowledgeGraphFacade`](../src/knowledge_graph/facade.py), [`GraphExpansion`](../src/knowledge_graph/expansion.py) (primary), [`GraphTraversal`](../src/knowledge_graph/traversal.py) (secondary), overlay/context/builder/loader/persist
- Colab helpers: [`retrieval.colab_runtime`](../src/retrieval/colab_runtime.py)
- Generation: remote OpenAI-compatible API only for Colab-safe RAM guarantees

**Primary vs secondary graph paths**
- **Primary:** vector-first hybrid expansion
- **Secondary (off under Colab-safe):** graph-guided pre-filter

Pure vector profiles remain usable if the graph is missing. Hybrid mode fails clearly when the graph is unavailable rather than silently falling back under a hybrid label.

**Full Graph Module coverage**
Stage C imports the entire public `knowledge_graph` surface and wires live services (`GraphLoader`, `GraphBuilder`, `GraphTraversal`, `OverlayJoiner`, `ContextBuilder`, `GraphExpansion`, persist helpers). See §4.4 demos after hybrid load.

**Note:** Demonstration notebook — not a replacement for [`scripts/verify_kg.py`](../scripts/verify_kg.py) or [`scripts/evaluate_e2e.py`](../scripts/evaluate_e2e.py).


## 1. Environment setup (before Stage A)


In [ ]:
# Optional: install runtime dependencies if your environment does not have them yet.
# Uncomment and run once if needed.
%pip install -q faiss-cpu sentence-transformers pandas openai


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import logging

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Useful if the notebook is launched from notebooks/.
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('HF Hub token detected in environment.')
else:
    logging.getLogger('huggingface_hub.utils._http').setLevel(logging.ERROR)
    print('HF_TOKEN not set; suppressing the Hugging Face unauthenticated-request warning.')

print('Project root:', PROJECT_ROOT)
print('src on path:', SRC_DIR.exists())


Project root: d:\Uni_Project\Text_Mining\Project
src on path: True


## 2. Configure artifact paths and retrieval settings (Stage A)

Change `INDEX_DIR` if you download the FAISS files somewhere else. Set `RUNTIME_PROFILE` to `colab_safe` (default) or `unconstrained`.


In [ ]:
# === Stage A: paths + runtime profile (005-colab-ram-fit) ===
# Directory containing index.faiss + payloads.jsonl (+ optional id_map.json)
INDEX_DIR = PROJECT_ROOT / 'data' / 'faiss_index'

# Graph sources
V2_DATA_DIR = PROJECT_ROOT / 'data' / 'v2'
GRAPH_PICKLE_PATH = PROJECT_ROOT / 'data' / 'graph' / 'knowledge_graph.gpickle'

# Runtime profile: "colab_safe" (~12GB) | "unconstrained" (local/high-RAM)
RUNTIME_PROFILE = 'colab_safe'

# Must match the embedding model used to build index.faiss.
# Alias EMBEDDING_MODEL_NAME kept for plan/quickstart naming parity.
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
EMBEDDING_MODEL_NAME = EMBEDDING_MODEL

# Embedder device only (query encoding). FAISS search stays on CPU (faiss-cpu).
# 'auto' = cuda if available else cpu; or force 'cuda' / 'cpu' / 'mps'.
EMBEDDER_DEVICE = 'auto'

# --- Profile resolution (conservative caps under colab_safe) ---
from retrieval.colab_runtime import (
    CleanupRequest,
    ResidentComponentSnapshot,
    apply_cleanup,
    build_load_plan,
    capture_memory_snapshot,
    decide_graph_source_mode,
    format_load_plan,
    format_resident_snapshot,
    format_session_outcome,
    payload_cache_rebuild_warning,
    resolve_runtime_profile,
    session_outcome_label,
)

# Optional explicit overrides (None → profile defaults)
_ALLOW_JSONL_GRAPH_REBUILD = False  # Colab-safe default; set True only after RAM warning
_RUN_PAYLOAD_CSV_EXPORT = None      # None → False under colab_safe, True under unconstrained
_RUN_PAYLOAD_CACHE_EXPORT = None
_RUN_BENCHMARK_SAMPLE = False
_RUN_FILTER_PROFILE_COMPARISON = None
_ENABLE_GRAPH_GUIDED_PREFILTER_DEMO = False

runtime_profile = resolve_runtime_profile(
    RUNTIME_PROFILE,
    project_root=PROJECT_ROOT,
    graph_pickle_path=GRAPH_PICKLE_PATH,
    v2_data_dir=V2_DATA_DIR,
    embedding_model=EMBEDDING_MODEL,
    allow_jsonl_graph_rebuild=_ALLOW_JSONL_GRAPH_REBUILD,
    enable_graph_guided_prefilter_demo=_ENABLE_GRAPH_GUIDED_PREFILTER_DEMO,
    run_payload_csv_export=_RUN_PAYLOAD_CSV_EXPORT,
    run_payload_cache_export=_RUN_PAYLOAD_CACHE_EXPORT,
    run_benchmark_sample=_RUN_BENCHMARK_SAMPLE,
    run_filter_profile_comparison=_RUN_FILTER_PROFILE_COMPARISON,
)

COLAB_SAFE = runtime_profile.colab_safe
ALLOW_JSONL_GRAPH_REBUILD = runtime_profile.allow_jsonl_graph_rebuild
TOP_K = runtime_profile.top_k
TOP_N = runtime_profile.top_n
SCORE_THRESHOLD = runtime_profile.score_threshold
HYBRID_MAX_HOP = runtime_profile.hybrid_max_hop
HYBRID_MAX_CONTEXT = runtime_profile.hybrid_max_context
EXPAND_UNITS = runtime_profile.local_expand_units
LOCAL_EXPAND_UNITS = EXPAND_UNITS
DEFAULT_FILTER_PROFILE = 'broad'  # current_law | broad | historical (non-graph)
FILTER_PROFILE = DEFAULT_FILTER_PROFILE
BENCHMARK_SAMPLE_SIZE = runtime_profile.benchmark_sample_size

# Hybrid graph settings
ENABLE_HYBRID_EXPANSION = runtime_profile.enable_hybrid_expansion
AS_OF_DATE = '2026-07-13'
USE_HYBRID_EVIDENCE_FOR_GENERATION = runtime_profile.use_hybrid_evidence_for_generation
ENABLE_GRAPH_GUIDED_PREFILTER_DEMO = runtime_profile.enable_graph_guided_prefilter_demo
# Full Graph Module demos (inventory + traversal modes + EvidenceContext + overlay sample)
# Lightweight when pickle-loaded; JSONL parse/build still gated by ALLOW_JSONL_GRAPH_REBUILD.
ENABLE_FULL_GRAPH_MODULE_DEMO = True

GRAPH_GUIDED_START_ID = ''  # optional document id_str; empty → take from first seed hit
GRAPH_GUIDED_TRAVERSAL_MODE = 'basis'
GRAPH_GUIDED_MAX_DEPTH = 2

# Heavy optional Stage E gates (default False under colab_safe)
RUN_PAYLOAD_CSV_EXPORT = runtime_profile.run_payload_csv_export
RUN_PAYLOAD_CACHE_EXPORT = runtime_profile.run_payload_cache_export
RUN_BENCHMARK_SAMPLE = runtime_profile.run_benchmark_sample
RUN_FILTER_PROFILE_COMPARISON = runtime_profile.run_filter_profile_comparison
PAYLOAD_CSV_EXPORT_LIMIT = runtime_profile.payload_csv_export_limit

print('RUNTIME_PROFILE:', RUNTIME_PROFILE, '| COLAB_SAFE:', COLAB_SAFE)
print('INDEX_DIR:', INDEX_DIR)
print('V2_DATA_DIR:', V2_DATA_DIR)
print('GRAPH_PICKLE_PATH:', GRAPH_PICKLE_PATH)
print('ALLOW_JSONL_GRAPH_REBUILD:', ALLOW_JSONL_GRAPH_REBUILD)
print('TOP_K/TOP_N/HYBRID_MAX_CONTEXT:', TOP_K, TOP_N, HYBRID_MAX_CONTEXT)
print('ENABLE_HYBRID_EXPANSION:', ENABLE_HYBRID_EXPANSION)
print('USE_HYBRID_EVIDENCE_FOR_GENERATION:', USE_HYBRID_EVIDENCE_FOR_GENERATION)
print('LOCAL_EXPAND_UNITS / EXPAND_UNITS:', LOCAL_EXPAND_UNITS)
print('ENABLE_GRAPH_GUIDED_PREFILTER_DEMO:', ENABLE_GRAPH_GUIDED_PREFILTER_DEMO)
print('ENABLE_FULL_GRAPH_MODULE_DEMO:', ENABLE_FULL_GRAPH_MODULE_DEMO)
print(
    'Heavy gates CSV/CACHE/BENCH/FILTER:',
    RUN_PAYLOAD_CSV_EXPORT,
    RUN_PAYLOAD_CACHE_EXPORT,
    RUN_BENCHMARK_SAMPLE,
    RUN_FILTER_PROFILE_COMPARISON,
)
INDEX_DIR


## 3. Preflight + load plan (Stage A)

Lists required FAISS artifacts, approximate sizes, graph source mode (`pickle` | `jsonl_rebuild` | `unavailable`), planned load/skip/opt-in actions, and best-effort memory snapshot.


In [ ]:
# ### COLAB_SAFE_RAM_FIT — Stage A load plan + memory preflight
phase_t0 = time.perf_counter()
required_files = [INDEX_DIR / 'index.faiss', INDEX_DIR / 'payloads.jsonl']
optional_files = [INDEX_DIR / 'id_map.json', INDEX_DIR / 'payload_cache.sqlite']

missing = [p for p in required_files if not p.exists()]
if missing:
    print('Downloads are not ready yet. Missing:')
    for p in missing:
        print(' -', p)
else:
    print('Required FAISS files found.')
    for p in required_files + optional_files:
        if p.exists():
            print(f'{p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB')
        else:
            print(f'{p.name}: MISSING')

mem_preflight = capture_memory_snapshot(note='preflight')
load_plan = build_load_plan(runtime_profile, index_dir=INDEX_DIR, memory_before=mem_preflight)
print(format_load_plan(load_plan))
print(f'Preflight cell finished in {time.perf_counter() - phase_t0:.2f}s')


## 4. Load the FAISS store and build retriever (Stage B)

Loads FAISS + preferred `payload_cache.sqlite` + embedder. Pure vector queries work after this stage **without** loading the graph.


In [ ]:
# === Stage B: load FAISS store + embedder + vector retriever ===
if missing:
    raise FileNotFoundError('Download index.faiss and payloads.jsonl before running this cell.')

from retrieval.config import VectorIndexConfig
from retrieval.embeddings import SentenceTransformerEmbedder
from retrieval.retriever import VectorRetriever
from retrieval.sqlite_faiss_store import SQLitePayloadFaissVectorStore

# FR-020: warn before costly cold payload cache rebuild on Colab
_cache_warn = payload_cache_rebuild_warning(INDEX_DIR)
if _cache_warn:
    print(_cache_warn)

load_t0 = time.perf_counter()
mem_before_vector = capture_memory_snapshot(note='before_vector_load')
print(mem_before_vector.format_line())

config = VectorIndexConfig(
    embedding_model=EMBEDDING_MODEL,
    top_k=TOP_K,
    top_n=TOP_N,
    score_threshold=SCORE_THRESHOLD,
    expand_units=EXPAND_UNITS,
)

store = SQLitePayloadFaissVectorStore.load(INDEX_DIR)

# Resolve embedder device (GPU for model weights/encoding only; FAISS remains CPU).
_requested_device = str(globals().get('EMBEDDER_DEVICE', 'auto')).strip().lower()
if _requested_device in ('', 'auto'):
    _resolved_device = 'cpu'
    try:
        import torch
        if torch.cuda.is_available():
            _resolved_device = 'cuda'
        elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
            _resolved_device = 'mps'
    except Exception:
        _resolved_device = 'cpu'
else:
    _resolved_device = _requested_device

if _resolved_device == 'cuda':
    try:
        import torch
        if not torch.cuda.is_available():
            print('EMBEDDER_DEVICE=cuda requested but CUDA unavailable; falling back to cpu')
            _resolved_device = 'cpu'
    except Exception as exc:
        print(f'CUDA check failed ({exc}); falling back to cpu')
        _resolved_device = 'cpu'

embedder = SentenceTransformerEmbedder(
    EMBEDDING_MODEL,
    query_prefix=config.query_prefix,
    passage_prefix=config.passage_prefix,
    device=_resolved_device,
)
retriever = VectorRetriever(config=config, embedder=embedder, store=store)

print(f'Vector retriever ready in {time.perf_counter() - load_t0:.2f}s')
print(f'Loaded FAISS vectors: {store.total_vectors:,}')
print(f'Embedding dimension: {embedder.dimension}')
print(f'Embedder device: {_resolved_device} (requested={_requested_device})')
print('FAISS backend: CPU (faiss-cpu / Index search on host RAM)')
print('Store class:', type(store).__module__ + '.' + type(store).__name__)
print('CPU-only path valid for Colab-safe success (FR-022); GPU optional for embedder only.')
if _resolved_device == 'cuda':
    try:
        import torch
        print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    except Exception:
        pass

# Vector-only retriever (graph_expansion=None). Hybrid retriever is wired after graph load.
hybrid_retriever = None
graph_expansion = None

mem_after_vector = capture_memory_snapshot(note='after_vector_load')
print(mem_after_vector.format_line())
print(
    format_resident_snapshot(
        ResidentComponentSnapshot(
            store_loaded=store is not None,
            embedder_loaded=embedder is not None,
            structural_graph_loaded=False,
            graph_source_mode=None,
            overlays_loaded=False,
            hybrid_retriever_ready=False,
            generator_configured=False,
            optional_frames_held=[],
        )
    )
)
print('Stage B complete: pure vector queries are usable without Stage C graph load.')
print(
    format_session_outcome(
        session_outcome_label(
            colab_safe=COLAB_SAFE,
            structural_ready=False,
            loaded_from_pickle=False,
            hybrid_used=False,
            vector_ok=True,
        )
    )
)


## 4.3 Hybrid graph integration (vector-first) — Stage C

Primary hybrid path (default full pipeline when graph loads):

```text
query → embed → vector seed retrieve → resolve chunk→provision→document
      → GraphExpansion + validity/authority overlays → fused evidence → optional LLM
```

**Colab-safe graph source policy**
1. Prefer `data/graph/knowledge_graph.gpickle` via `load_knowledge_graph`
2. JSONL rebuild only if `ALLOW_JSONL_GRAPH_REBUILD=True` (warns under Colab-safe)
3. Else hybrid unavailable; pure vector remains usable

Secondary path (optional only, **off** under Colab-safe): graph-guided pre-filter whitelist before vector search.

**Full Graph Module orchestration (this notebook)**
Stage C imports and wires the entire `src/knowledge_graph/` surface through [`KnowledgeGraphFacade`](../src/knowledge_graph/facade.py) plus direct services:

| Layer | Modules | Notebook role |
| --- | --- | --- |
| Loader | `loader.py` | path contract + optional JSONL source streams |
| Parser / edges | `parser.py`, `schema.py`, `edge_parser.py`, `edge_schema.py` | typed nodes/edges (via facade parse / rebuild) |
| Builder | `builder.py` | structural `KnowledgeGraph` (pickle load or JSONL rebuild) |
| Persist | `persist.py` | preferred Colab-safe pickle load |
| Expansion | `expansion.py`, `expansion_schema.py` | **primary** hybrid seed→context |
| Traversal | `traversal.py` | **secondary** modes + guided pre-filter |
| Overlay | `overlay.py`, `overlay_schema.py` | validity/authority join (non-mutating) |
| Context | `context.py`, `context_schema.py` | guided filter + `EvidenceContext` |
| Facade | `facade.py` | public orchestration |
| Utils | `utils.py` | coercion helpers used by parsers/overlays |

This section orchestrates existing modules under `src/knowledge_graph/` and `src/retrieval/` — it does **not** reimplement graph logic.


In [ ]:
# ### HYBRID_GRAPH_INTEGRATION — intended import surface (FR-002)
# ### FULL_GRAPH_MODULE — full knowledge_graph package surface
# ### COLAB_SAFE_RAM_FIT — pickle load surface (004)
from dataclasses import dataclass, field
from typing import Any, Literal

# --- Public package surface (loader → parse → build → traverse → expand → overlay → context → persist) ---
from knowledge_graph import (
    # loader
    GraphLoader,
    GraphLoaderPaths,
    GraphSourceBundle,
    load_jsonl_records,
    # schema / parser nodes
    ChunkNode,
    DocumentNode,
    ExternalStubNode,
    FacetValue,
    ProvisionNode,
    TextProvenanceRecord,
    index_text_provenance,
    parse_chunk_row,
    parse_chunk_rows,
    parse_document_row,
    parse_document_rows,
    parse_external_stub_row,
    parse_external_stub_rows,
    parse_provision_row,
    parse_provision_rows,
    parse_text_provenance_row,
    # edges
    GraphEdge,
    parse_edge_row,
    parse_edge_rows,
    verified_edge_rows,
    # builder
    GraphBuildResult,
    GraphBuildStats,
    GraphBuilder,
    KnowledgeGraph,
    StructuralEdge,
    # traversal
    GraphTraversal,
    TraversalMode,
    TraversalPath,
    TraversalResult,
    TraversalStep,
    # expansion (primary hybrid)
    GraphExpansion,
    ExpansionResult,
    ExpansionStep,
    # overlay
    OverlayBundle,
    OverlayJoiner,
    AuthorityIndexEntry,
    DocumentOverlay,
    ValidityEvent,
    compute_currency_status,
    index_authority_index,
    index_validity_timeline,
    parse_authority_index_row,
    parse_authority_index_rows,
    parse_validity_event_row,
    parse_validity_event_rows,
    resolve_authority_rank_conflicts,
    # context
    ContextBuilder,
    EvidenceContext,
    FilterProfile,
    GraphGuidedFilter,
    QueryConstraints,
    # facade + parse bundle
    KnowledgeGraphFacade,
    ParsedGraphSources,
    # persist
    FORMAT_NAME,
    FORMAT_VERSION,
    GraphPickleArtifactInfo,
    GraphPickleCorruptError,
    GraphPickleEnvelope,
    GraphPickleError,
    GraphPickleIncompatibleError,
    GraphPickleLoadResult,
    GraphPickleNotFoundError,
    load_knowledge_graph,
    save_knowledge_graph,
)

# Internal helpers (not re-exported on package __all__, still part of the module)
from knowledge_graph import utils as kg_utils
from knowledge_graph.utils import as_bool, as_int, quality_flags

from retrieval.io_utils import read_jsonl
from retrieval.schema import RetrievedChunk, RetrievalResult
from retrieval.stores import SearchHit

GRAPH_MODULE_SURFACE = {
    'loader': (GraphLoader, GraphLoaderPaths, GraphSourceBundle, load_jsonl_records),
    'parser_schema': (
        ChunkNode, DocumentNode, ExternalStubNode, FacetValue, ProvisionNode,
        TextProvenanceRecord, index_text_provenance,
        parse_chunk_rows, parse_document_rows, parse_external_stub_rows,
        parse_provision_rows, parse_text_provenance_row,
    ),
    'edge_parser': (GraphEdge, parse_edge_row, parse_edge_rows, verified_edge_rows),
    'builder': (GraphBuilder, KnowledgeGraph, GraphBuildResult, GraphBuildStats, StructuralEdge),
    'traversal': (GraphTraversal, TraversalMode, TraversalPath, TraversalResult, TraversalStep),
    'expansion': (GraphExpansion, ExpansionResult, ExpansionStep),
    'overlay': (
        OverlayJoiner, OverlayBundle, DocumentOverlay, ValidityEvent, AuthorityIndexEntry,
        parse_validity_event_rows, parse_authority_index_rows,
        index_validity_timeline, index_authority_index,
        compute_currency_status, resolve_authority_rank_conflicts,
    ),
    'context': (ContextBuilder, EvidenceContext, GraphGuidedFilter, QueryConstraints, FilterProfile),
    'facade': (KnowledgeGraphFacade, ParsedGraphSources),
    'persist': (
        load_knowledge_graph, save_knowledge_graph, GraphPickleLoadResult,
        FORMAT_NAME, FORMAT_VERSION, GraphPickleError,
    ),
    'utils': (as_int, as_bool, quality_flags, kg_utils),
}

print('Full Graph Module import surface ready:')
for group, objs in GRAPH_MODULE_SURFACE.items():
    names = ', '.join(getattr(o, '__name__', type(o).__name__) for o in objs[:4])
    more = f' (+{len(objs)-4} more)' if len(objs) > 4 else ''
    print(f'  [{group}] {names}{more}')


In [ ]:
# ### HYBRID_GRAPH_INTEGRATION — preflight, pickle-prefer load, overlays, expansion wire, guard
# ### COLAB_SAFE_RAM_FIT — Stage C graph load policy (FR-003/FR-004)


@dataclass
class GraphLoadStatus:
    structural_ready: bool = False
    overlays_ready: bool = False
    missing_structural_files: list[str] = field(default_factory=list)
    missing_overlay_files: list[str] = field(default_factory=list)
    build_stats: Any = None
    build_warnings: tuple[str, ...] = ()
    as_of_date: str | None = None
    overlay_coverage: dict[str, int] | None = None
    error: str | None = None
    build_duration_s: float | None = None
    # 005 extensions
    graph_source_mode: Literal['pickle', 'jsonl_rebuild', 'unavailable'] = 'unavailable'
    pickle_path: str | None = None
    loaded_from_pickle: bool = False
    rebuild_opt_in_required: bool = False
    rebuild_warning_emitted: bool = False


def preflight_graph_sources(
    v2_dir: Path,
    *,
    pickle_path: Path | None = None,
    colab_safe: bool = True,
    allow_jsonl_rebuild: bool = False,
) -> GraphLoadStatus:
    """Inventory structural/overlay files + decide graph source mode (FR-003/FR-004)."""
    paths = GraphLoaderPaths(data_dir=v2_dir)
    missing_structural = [str(p) for p in paths.required_paths() if not p.exists()]
    overlay_names = ('validity_timeline.jsonl', 'authority_index.jsonl')
    missing_overlay = [str(v2_dir / name) for name in overlay_names if not (v2_dir / name).exists()]

    decision = decide_graph_source_mode(
        pickle_path=pickle_path or GRAPH_PICKLE_PATH,
        v2_data_dir=v2_dir,
        colab_safe=colab_safe,
        allow_jsonl_graph_rebuild=allow_jsonl_rebuild,
        prefer_graph_pickle=True,
    )

    status = GraphLoadStatus(
        structural_ready=False,  # set True only after successful load
        overlays_ready=not missing_overlay,
        missing_structural_files=missing_structural,
        missing_overlay_files=missing_overlay,
        graph_source_mode=decision.mode,
        pickle_path=str(decision.pickle_path) if decision.pickle_path else None,
        loaded_from_pickle=False,
        rebuild_opt_in_required=decision.rebuild_opt_in_required,
        rebuild_warning_emitted=bool(decision.rebuild_warning),
    )

    print('=== Graph preflight (Colab-safe policy) ===')
    print('V2_DATA_DIR:', v2_dir)
    print('GRAPH_PICKLE_PATH:', pickle_path or GRAPH_PICKLE_PATH)
    print('graph_source_mode:', decision.mode)
    print('pickle present:', decision.pickle_present)
    print('jsonl structural ready:', decision.jsonl_structural_ready)
    print('rebuild_opt_in_required:', decision.rebuild_opt_in_required)
    if decision.rebuild_warning:
        print('WARNING:', decision.rebuild_warning)
        status.rebuild_warning_emitted = True

    print('Structural JSONL files:')
    for p in paths.required_paths():
        flag = 'OK' if p.exists() else 'MISSING'
        print(f'  [{flag}] {p}')
    print('Overlay files (optional):')
    for name in overlay_names:
        p = v2_dir / name
        flag = 'OK' if p.exists() else 'MISSING'
        print(f'  [{flag}] {p}')

    if decision.mode == 'unavailable':
        print('Structural graph UNAVAILABLE under current policy. Pure vector profiles remain usable.')
        print('Detail:', decision.detail)
        if decision.missing_structural_jsonl:
            print('Missing structural JSONL:')
            for m in decision.missing_structural_jsonl:
                print(' -', m)
    if missing_overlay:
        print('Overlays unavailable (currency/authority labeled unavailable if structural graph loads).')
        for m in missing_overlay:
            print(' -', m)
    return status


graph_load_status = preflight_graph_sources(
    V2_DATA_DIR,
    pickle_path=GRAPH_PICKLE_PATH,
    colab_safe=COLAB_SAFE,
    allow_jsonl_rebuild=ALLOW_JSONL_GRAPH_REBUILD,
)
kg_facade: KnowledgeGraphFacade | None = None
kg_graph = None
kg_build_result = None
kg_pickle_result: GraphPickleLoadResult | None = None
overlay_bundle: OverlayBundle | None = None
graph_expansion: GraphExpansion | None = None
hybrid_retriever: VectorRetriever | None = None
document_overlays: dict[str, DocumentOverlay] = {}

mem_before_graph = capture_memory_snapshot(note='before_graph_load')
print(mem_before_graph.format_line())

if graph_load_status.graph_source_mode == 'pickle':
    try:
        print('\n=== Graph load (portable pickle preferred) ===')
        kg_paths = GraphLoaderPaths(data_dir=V2_DATA_DIR)
        kg_facade = KnowledgeGraphFacade(paths=kg_paths)
        build_t0 = time.perf_counter()
        kg_pickle_result = load_knowledge_graph(GRAPH_PICKLE_PATH)
        duration = time.perf_counter() - build_t0
        kg_graph = kg_pickle_result.graph
        stats = kg_pickle_result.stats
        graph_load_status.build_stats = stats
        graph_load_status.build_warnings = tuple(kg_pickle_result.warnings or ())
        graph_load_status.build_duration_s = duration
        graph_load_status.structural_ready = True
        graph_load_status.loaded_from_pickle = True
        graph_load_status.graph_source_mode = 'pickle'
        graph_load_status.pickle_path = str(GRAPH_PICKLE_PATH)
        print(f'Knowledge graph loaded from pickle in {duration:.2f}s')
        print('format_version:', kg_pickle_result.format_version)
        if stats is not None:
            print('[Pickle / build statistics]')
            for attr in (
                'document_count',
                'external_stub_count',
                'provision_count',
                'chunk_count',
                'document_edge_count',
                'verified_document_edge_count',
                'unverified_document_edge_count',
                'structural_edge_count',
                'orphan_provision_count',
                'orphan_chunk_count',
            ):
                if hasattr(stats, attr):
                    print(f'  {attr}: {getattr(stats, attr):,}')
                elif isinstance(stats, dict) and attr in stats:
                    print(f'  {attr}: {stats[attr]:,}')
        if kg_pickle_result.warnings:
            print('Load warnings:')
            for w in list(kg_pickle_result.warnings)[:20]:
                print(' -', w)

        # Overlays (optional — never required for structural hybrid success)
        if not graph_load_status.missing_overlay_files:
            print('\n=== Overlay join ===')
            validity_path = V2_DATA_DIR / 'validity_timeline.jsonl'
            authority_path = V2_DATA_DIR / 'authority_index.jsonl'
            validity_events = list(parse_validity_event_rows(read_jsonl(validity_path)))
            authority_entries = list(parse_authority_index_rows(read_jsonl(authority_path)))
            overlay_bundle = kg_facade.build_overlay_bundle(
                documents=kg_graph.documents.values(),
                validity_events=validity_events,
                authority_entries=authority_entries,
                as_of_date=AS_OF_DATE,
            )
            document_overlays = dict(overlay_bundle.document_overlays)
            graph_load_status.overlays_ready = True
            graph_load_status.as_of_date = AS_OF_DATE
            currency_hist: dict[str, int] = {}
            for ov in document_overlays.values():
                currency_hist[ov.currency_status] = currency_hist.get(ov.currency_status, 0) + 1
            graph_load_status.overlay_coverage = currency_hist
            print(f'Overlays joined for {len(document_overlays):,} documents (as_of={AS_OF_DATE})')
            print('currency_status histogram:', currency_hist)
        else:
            graph_load_status.overlays_ready = False
            print('Overlays MISSING — structural expansion allowed; currency/authority labeled unavailable.')

        graph_expansion = GraphExpansion(kg_graph)
        hybrid_retriever = VectorRetriever(
            config=config,
            embedder=embedder,
            store=store,
            graph_expansion=graph_expansion,
        )
        print('\nGraphExpansion wired. hybrid_retriever has graph_expansion; vector-only retriever remains graph_expansion=None.')
        print('Label reminder: graph_expansion ≠ local_expand_units')
        # FULL_GRAPH_MODULE service handles (loader/builder/traversal/overlay/context)
        # are attached in the following full-module cell once this load succeeds.
        print('Full Graph Module services will attach in the next Stage C cell.')
    except Exception as exc:
        graph_load_status.structural_ready = False
        graph_load_status.loaded_from_pickle = False
        graph_load_status.graph_source_mode = 'unavailable'
        graph_load_status.error = str(exc)
        kg_facade = None
        kg_graph = None
        graph_expansion = None
        hybrid_retriever = None
        print('Graph pickle load FAILED:', exc)
        print('Pure vector retrieval remains usable; hybrid mode will fail clearly if requested.')

elif graph_load_status.graph_source_mode == 'jsonl_rebuild':
    try:
        print('\n=== Graph build (JSONL rebuild — opt-in / unconstrained) ===')
        if COLAB_SAFE:
            print(
                'WARNING: Full structural JSONL rebuild may exceed ~12GB RAM. '
                'ALLOW_JSONL_GRAPH_REBUILD=True acknowledged.'
            )
            graph_load_status.rebuild_warning_emitted = True
        kg_paths = GraphLoaderPaths(data_dir=V2_DATA_DIR)
        kg_facade = KnowledgeGraphFacade(paths=kg_paths)
        build_t0 = time.perf_counter()
        kg_build_result = kg_facade.build_graph()
        duration = time.perf_counter() - build_t0
        kg_graph = kg_build_result.graph
        stats = kg_build_result.stats
        graph_load_status.build_stats = stats
        graph_load_status.build_warnings = tuple(kg_build_result.warnings or ())
        graph_load_status.build_duration_s = duration
        graph_load_status.structural_ready = True
        graph_load_status.loaded_from_pickle = False
        graph_load_status.graph_source_mode = 'jsonl_rebuild'
        print(f'Knowledge graph built from JSONL in {duration:.2f}s')
        print('[Build Statistics]')
        print(f'  documents:           {stats.document_count:,}')
        print(f'  external_stubs:      {stats.external_stub_count:,}')
        print(f'  provisions:          {stats.provision_count:,}')
        print(f'  chunks:              {stats.chunk_count:,}')
        print(f'  document_edges:      {stats.document_edge_count:,}')
        print(f'  verified_edges:      {stats.verified_document_edge_count:,}')
        print(f'  unverified_edges:    {stats.unverified_document_edge_count:,}')
        print(f'  structural_edges:    {stats.structural_edge_count:,}')
        print(f'  orphan_provisions:   {stats.orphan_provision_count}')
        print(f'  orphan_chunks:       {stats.orphan_chunk_count}')
        if kg_build_result.warnings:
            print('Build warnings:')
            for w in kg_build_result.warnings[:20]:
                print(' -', w)

        if not graph_load_status.missing_overlay_files:
            print('\n=== Overlay join ===')
            validity_path = V2_DATA_DIR / 'validity_timeline.jsonl'
            authority_path = V2_DATA_DIR / 'authority_index.jsonl'
            validity_events = list(parse_validity_event_rows(read_jsonl(validity_path)))
            authority_entries = list(parse_authority_index_rows(read_jsonl(authority_path)))
            overlay_bundle = kg_facade.build_overlay_bundle(
                documents=kg_graph.documents.values(),
                validity_events=validity_events,
                authority_entries=authority_entries,
                as_of_date=AS_OF_DATE,
            )
            document_overlays = dict(overlay_bundle.document_overlays)
            graph_load_status.overlays_ready = True
            graph_load_status.as_of_date = AS_OF_DATE
            currency_hist = {}
            for ov in document_overlays.values():
                currency_hist[ov.currency_status] = currency_hist.get(ov.currency_status, 0) + 1
            graph_load_status.overlay_coverage = currency_hist
            print(f'Overlays joined for {len(document_overlays):,} documents (as_of={AS_OF_DATE})')
            print('currency_status histogram:', currency_hist)
        else:
            graph_load_status.overlays_ready = False
            print('Overlays MISSING — structural expansion allowed; currency/authority labeled unavailable.')

        graph_expansion = GraphExpansion(kg_graph)
        hybrid_retriever = VectorRetriever(
            config=config,
            embedder=embedder,
            store=store,
            graph_expansion=graph_expansion,
        )
        print('\nGraphExpansion wired after JSONL rebuild.')
        print('Full Graph Module services will attach in the next Stage C cell.')
    except Exception as exc:
        graph_load_status.structural_ready = False
        graph_load_status.loaded_from_pickle = False
        graph_load_status.graph_source_mode = 'unavailable'
        graph_load_status.error = str(exc)
        kg_facade = None
        kg_graph = None
        graph_expansion = None
        hybrid_retriever = None
        print('Graph JSONL rebuild FAILED:', exc)
        print('Pure vector retrieval remains usable; hybrid mode will fail clearly if requested.')
else:
    print('Skipping graph load (pickle missing and/or JSONL rebuild not permitted under Colab-safe).')
    print('Hybrid labeled path unavailable; pure vector retrieval remains usable.')
    if graph_load_status.rebuild_opt_in_required:
        print('To rebuild from JSONL: set ALLOW_JSONL_GRAPH_REBUILD=True (may exceed 12GB) and re-run this cell.')


def require_graph_for_hybrid(action: str = 'hybrid expansion') -> None:
    """Fail clearly if hybrid is requested without a loaded graph (FR-006 / no silent fallback)."""
    if not graph_load_status.structural_ready or kg_graph is None or graph_expansion is None:
        missing = graph_load_status.missing_structural_files or ['(structural graph not loaded)']
        detail = graph_load_status.error or graph_load_status.graph_source_mode or '; '.join(missing)
        raise RuntimeError(
            f"Cannot run {action}: knowledge graph unavailable (hybrid_unavailable). "
            f"Do not silently fall back to vector-only under a hybrid label. "
            f"graph_source_mode={graph_load_status.graph_source_mode}. Detail: {detail}"
        )


mem_after_graph = capture_memory_snapshot(note='after_graph_load')
print(mem_after_graph.format_line())
print('\nGraphLoadStatus:')
print('  structural_ready:', graph_load_status.structural_ready)
print('  overlays_ready:', graph_load_status.overlays_ready)
print('  graph_source_mode:', graph_load_status.graph_source_mode)
print('  loaded_from_pickle:', graph_load_status.loaded_from_pickle)
print('  pickle_path:', graph_load_status.pickle_path)
print('  rebuild_opt_in_required:', graph_load_status.rebuild_opt_in_required)
print('  error:', graph_load_status.error)
print(
    format_resident_snapshot(
        ResidentComponentSnapshot(
            store_loaded='store' in globals() and store is not None,
            embedder_loaded='embedder' in globals() and embedder is not None,
            structural_graph_loaded=bool(graph_load_status.structural_ready and kg_graph is not None),
            graph_source_mode=graph_load_status.graph_source_mode,
            overlays_loaded=bool(graph_load_status.overlays_ready),
            hybrid_retriever_ready=hybrid_retriever is not None,
            generator_configured=bool(globals().get('generator')),
            optional_frames_held=[],
        )
    )
)
if graph_load_status.structural_ready and graph_load_status.loaded_from_pickle:
    print(
        format_session_outcome(
            'hybrid_colab_safe_success_pickle'
            if COLAB_SAFE
            else 'unconstrained_success',
            detail='Graph loaded; run hybrid smoke/demo next.',
        )
    )
elif not graph_load_status.structural_ready:
    print(format_session_outcome('hybrid_unavailable', detail='Vector-only remains usable.'))


In [ ]:
# ### HYBRID_GRAPH_INTEGRATION — two-stage hybrid helper + views (US1)


@dataclass
class SeedRetrievalView:
    query: str
    filter_profile: str
    total_candidates: int
    seed_chunks: list[RetrievedChunk]
    seed_chunk_ids: list[str]
    mode_label: str  # vector_only | hybrid_seed


@dataclass
class GraphExpansionView:
    expansion: ExpansionResult | None
    expanded_chunk_ids: list[str]
    added_chunk_ids: list[str]
    resolved_chunks: list[RetrievedChunk]
    warnings: list[str]
    capped: bool
    mechanism_label: str = 'graph_expansion'


@dataclass
class HybridEvidenceContext:
    query: str
    mode: str  # vector_only | hybrid_expanded | graph_guided_prefilter
    seed: SeedRetrievalView
    expansion: GraphExpansionView | None
    evidence_chunks: list[RetrievedChunk]
    document_overlays: dict[str, DocumentOverlay]
    overlay_available: bool
    expansion_added_context: bool
    diagnostics: list[str] = field(default_factory=list)


@dataclass
class ModeComparisonRecord:
    query: str
    vector_only_count: int
    hybrid_count: int
    expansion_ran: bool
    added_context_count: int
    sample_vector_only_ids: list[str]
    sample_hybrid_ids: list[str]
    notes: list[str] = field(default_factory=list)


@dataclass
class GraphGuidedDemoResult:
    start_id: str
    traversal_mode: str
    whitelist_size: int
    empty_filter_warning: bool
    filter_reason: str
    retrieval: RetrievalResult | None


def _hit_to_retrieved_chunk(hit: SearchHit, query: str, filter_profile: str) -> RetrievedChunk:
    """Reuse VectorRetriever conversion so identity fields stay consistent."""
    return retriever._to_retrieved_chunk(hit, query, filter_profile)


def _is_citation_safe_chunk(chunk: RetrievedChunk) -> bool:
    """External stubs / non-citation-safe nodes are never citation-ready (FR-013)."""
    meta = chunk.metadata or {}
    if meta.get('is_external_stub') is True:
        return False
    if meta.get('citation_safe') is False:
        return False
    # Graph external stubs keyed by id_str
    if kg_graph is not None and chunk.id_str and chunk.id_str in getattr(kg_graph, 'external_stubs', {}):
        return False
    if not (chunk.chunk_text or '').strip():
        return False
    return True


def _resolve_chunk_ids(chunk_ids: list[str], query: str, filter_profile: str) -> list[RetrievedChunk]:
    if not chunk_ids:
        return []
    # Preserve order from expansion; scroll may return unordered.
    hits = store.scroll({'chunk_id': {'in': list(chunk_ids)}}, limit=max(len(chunk_ids), 1))
    by_id: dict[str, SearchHit] = {}
    for hit in hits:
        cid = str(hit.payload.get('chunk_id') or hit.point_id)
        by_id[cid] = hit
    ordered: list[RetrievedChunk] = []
    for cid in chunk_ids:
        hit = by_id.get(cid)
        if hit is None:
            continue
        ordered.append(_hit_to_retrieved_chunk(hit, query, filter_profile))
    return ordered


def run_hybrid_retrieve(
    query: str,
    *,
    top_n: int = TOP_N,
    filter_profile: str = FILTER_PROFILE,
    score_threshold: float | None = SCORE_THRESHOLD,
    enable_expansion: bool | None = None,
    max_hop: int | None = None,
    max_context: int | None = None,
) -> HybridEvidenceContext:
    """Vector seed → optional GraphExpansion → overlay join → HybridEvidenceContext."""
    do_expand = ENABLE_HYBRID_EXPANSION if enable_expansion is None else enable_expansion
    max_hop = HYBRID_MAX_HOP if max_hop is None else max_hop
    max_context = HYBRID_MAX_CONTEXT if max_context is None else max_context
    diagnostics: list[str] = []

    # Stage 1: seed vector retrieve (never local expand here)
    seed_result = retriever.retrieve(
        query,
        top_n=top_n,
        filter_profile=filter_profile,
        score_threshold=score_threshold,
        expand_units=False,
    )
    seed_chunks = list(seed_result.chunks)
    seed_ids = [c.chunk_id for c in seed_chunks]
    seed_view = SeedRetrievalView(
        query=query,
        filter_profile=seed_result.filter_profile_used,
        total_candidates=seed_result.total_candidates,
        seed_chunks=seed_chunks,
        seed_chunk_ids=seed_ids,
        mode_label='hybrid_seed' if do_expand else 'vector_only',
    )

    if not do_expand:
        diagnostics.append('Hybrid expansion disabled — returning vector-only seeds.')
        return HybridEvidenceContext(
            query=query,
            mode='vector_only',
            seed=seed_view,
            expansion=None,
            evidence_chunks=seed_chunks,
            document_overlays={},
            overlay_available=graph_load_status.overlays_ready,
            expansion_added_context=False,
            diagnostics=diagnostics,
        )

    require_graph_for_hybrid('hybrid expansion')

    if not seed_ids:
        diagnostics.append('Zero seed hits — skipping graph expansion and recording empty context.')
        empty_expansion = GraphExpansionView(
            expansion=None,
            expanded_chunk_ids=[],
            added_chunk_ids=[],
            resolved_chunks=[],
            warnings=['No seed chunk IDs; expansion skipped.'],
            capped=False,
            mechanism_label='graph_expansion',
        )
        return HybridEvidenceContext(
            query=query,
            mode='hybrid_expanded',
            seed=seed_view,
            expansion=empty_expansion,
            evidence_chunks=[],
            document_overlays={},
            overlay_available=graph_load_status.overlays_ready,
            expansion_added_context=False,
            diagnostics=diagnostics,
        )

    expansion_result = graph_expansion.expand(
        seed_ids,
        max_hop=max_hop,
        max_context=max_context,
    )
    expanded_ids = list(expansion_result.ordered_context_chunks)
    seed_set = set(seed_ids)
    added_ids = [cid for cid in expanded_ids if cid not in seed_set]
    # Prefer expanded order; if expansion returned nothing usable, fall back to seeds
    ordered_ids = expanded_ids or list(seed_ids)
    capped = bool(
        max_context is not None
        and expansion_result.max_context is not None
        and len(expanded_ids) >= int(expansion_result.max_context)
    )
    if capped:
        diagnostics.append(f'Expansion context capped at max_context={max_context}.')

    warnings = list(expansion_result.warnings or ())
    resolved = _resolve_chunk_ids(ordered_ids, query, filter_profile)
    # Keep citation-ready only for generation/display of "evidence"
    citation_ready = [c for c in resolved if _is_citation_safe_chunk(c)]
    dropped = len(resolved) - len(citation_ready)
    if dropped:
        diagnostics.append(f'Excluded {dropped} non-citation-safe/stub/empty chunks from citation-ready evidence.')

    if not added_ids:
        diagnostics.append('Graph expansion ran with zero added neighbors (seeds only) — not a failure.')
    else:
        diagnostics.append(f'Graph expansion added {len(added_ids)} chunk ids beyond seeds.')

    for w in warnings:
        diagnostics.append(f'Expansion warning: {w}')

    expansion_view = GraphExpansionView(
        expansion=expansion_result,
        expanded_chunk_ids=expanded_ids,
        added_chunk_ids=added_ids,
        resolved_chunks=resolved,
        warnings=warnings,
        capped=capped,
        mechanism_label='graph_expansion',
    )

    # Overlay join by id_str (display-only signals; do not mutate payloads)
    involved_ids = {c.id_str for c in citation_ready if c.id_str}
    subset_overlays: dict[str, DocumentOverlay] = {}
    if graph_load_status.overlays_ready and document_overlays:
        subset_overlays = {i: document_overlays[i] for i in involved_ids if i in document_overlays}
        diagnostics.append(f'Overlays attached for {len(subset_overlays)}/{len(involved_ids)} involved documents (as_of={AS_OF_DATE}).')
    else:
        diagnostics.append('Overlays unavailable — structural expansion only; no authoritative currency claims.')

    evidence = citation_ready if citation_ready else list(seed_chunks)
    return HybridEvidenceContext(
        query=query,
        mode='hybrid_expanded',
        seed=seed_view,
        expansion=expansion_view,
        evidence_chunks=evidence,
        document_overlays=subset_overlays,
        overlay_available=graph_load_status.overlays_ready,
        expansion_added_context=bool(added_ids),
        diagnostics=diagnostics,
    )


def chunks_to_display_rows(chunks: list[RetrievedChunk], *, limit: int | None = None) -> list[dict[str, Any]]:
    rows = []
    for rank, chunk in enumerate(chunks[: limit or len(chunks)], start=1):
        rows.append({
            'rank': rank,
            'chunk_id': chunk.chunk_id,
            'parent_unit_id': chunk.parent_unit_id,
            'id_str': chunk.id_str,
            'citation': chunk.citation_anchor or chunk.citation_label,
            'title': chunk.title,
            'validity_group': chunk.validity_group,
            'legal_authority_rank': chunk.legal_authority_rank,
            'vector_score': round(chunk.vector_score, 4),
            'text': (chunk.chunk_text or '')[:400],
        })
    return rows


print('Hybrid helper ready: run_hybrid_retrieve(), require_graph_for_hybrid(), chunks_to_display_rows()')


## 4.4 Full Graph Module surface demos (Stage C extension)

Opt-in inventory and lightweight demos for **every** graph layer after a successful structural load.

Controlled by `ENABLE_FULL_GRAPH_MODULE_DEMO` (default **True** when the structural graph is ready; still skips heavy JSONL parse/build under Colab-safe unless `ALLOW_JSONL_GRAPH_REBUILD=True`).

Demos (read-only / bounded):
1. Module inventory + live service handles (`loader`, `builder`, `traversal`, `overlay_joiner`, `context_builder`, `expansion`)
2. Direct [`GraphTraversal`](../src/knowledge_graph/traversal.py) modes (`basis` / `guidance` / `validity` / `structure` / `neighbors`)
3. [`ContextBuilder.build_evidence_context`](../src/knowledge_graph/context.py) + citation context
4. Direct [`OverlayJoiner`](../src/knowledge_graph/overlay.py) sample (when overlay files exist)
5. Optional facade `parse_sources` / `build_graph` **only** when JSONL rebuild is permitted

Primary hybrid path (`run_hybrid_retrieve` / `GraphExpansion`) remains unchanged.


In [ ]:
# ### FULL_GRAPH_MODULE — service handles + inventory + demos for all graph layers
# Keeps primary hybrid path intact; demos are read-only / bounded (005-safe).

# Live service handles populated after Stage C load (None when graph unavailable)
kg_loader: GraphLoader | None = None
kg_builder: GraphBuilder | None = None
kg_overlay_joiner: OverlayJoiner | None = None
kg_context_builder: ContextBuilder | None = None
kg_traversal: GraphTraversal | None = None
kg_parsed_sources: ParsedGraphSources | None = None
full_graph_module_demo: dict[str, Any] | None = None

if kg_facade is not None:
    kg_loader = kg_facade.loader
    kg_builder = kg_facade.builder
    kg_overlay_joiner = kg_facade.overlay_joiner
    kg_context_builder = kg_facade.context_builder
if kg_facade is not None and kg_graph is not None:
    kg_traversal = kg_facade.build_traversal(kg_graph)


@dataclass
class GraphModuleInventoryRow:
    layer: str
    module_file: str
    status: str
    detail: str


def inventory_graph_modules() -> list[GraphModuleInventoryRow]:
    """Report which graph layers are importable and which live services are wired."""
    rows: list[GraphModuleInventoryRow] = []

    def add(layer: str, module_file: str, ready: bool, detail: str) -> None:
        rows.append(
            GraphModuleInventoryRow(
                layer=layer,
                module_file=module_file,
                status='ready' if ready else 'not_wired',
                detail=detail,
            )
        )

    add('loader', 'loader.py', kg_loader is not None, type(kg_loader).__name__ if kg_loader else 'GraphLoader not attached')
    add('parser/schema', 'parser.py + schema.py', True, 'parse_* / node types imported on package surface')
    add('edge_parser', 'edge_parser.py + edge_schema.py', True, 'GraphEdge + parse_edge_rows imported')
    add('builder', 'builder.py', kg_builder is not None or kg_graph is not None,
        f'builder={type(kg_builder).__name__ if kg_builder else None}; graph_loaded={kg_graph is not None}')
    add('persist', 'persist.py', True, f'FORMAT={FORMAT_NAME} v{FORMAT_VERSION}; pickle_mode={graph_load_status.graph_source_mode}')
    add('expansion', 'expansion.py', graph_expansion is not None, 'primary hybrid path')
    add('traversal', 'traversal.py', kg_traversal is not None, 'direct GraphTraversal + facade.traverse')
    add('overlay', 'overlay.py', kg_overlay_joiner is not None,
        f'joiner_ready; overlays_ready={graph_load_status.overlays_ready}; n={len(document_overlays)}')
    add('context', 'context.py', kg_context_builder is not None, 'ContextBuilder + EvidenceContext / GraphGuidedFilter')
    add('facade', 'facade.py', kg_facade is not None, type(kg_facade).__name__ if kg_facade else 'missing')
    add('utils', 'utils.py', True, f'as_int/as_bool/quality_flags via knowledge_graph.utils')
    return rows


def print_graph_module_inventory() -> list[GraphModuleInventoryRow]:
    rows = inventory_graph_modules()
    print('=== Full Graph Module inventory ===')
    print(f'structural_ready={graph_load_status.structural_ready} source={graph_load_status.graph_source_mode}')
    for row in rows:
        print(f'  [{row.status:10}] {row.layer:14} | {row.module_file:28} | {row.detail}')
    return rows


def _pick_demo_start_id(preferred: str | None = None) -> str | None:
    """Choose a document id_str for traversal/context demos."""
    sid = (preferred or GRAPH_GUIDED_START_ID or '').strip()
    if sid:
        return sid
    if kg_graph is None:
        return None
    if kg_graph.verified_document_edges:
        for edge in kg_graph.verified_document_edges:
            if edge.src_id in kg_graph.documents:
                return edge.src_id
    if kg_graph.documents:
        return next(iter(kg_graph.documents.keys()))
    return None


def run_traversal_modes_demo(
    start_id: str | None = None,
    *,
    max_depth: int = 2,
    modes: tuple[str, ...] = ('basis', 'guidance', 'validity', 'structure', 'neighbors'),
) -> dict[str, TraversalResult]:
    """Exercise GraphTraversal directly (all modes) via facade.build_traversal."""
    require_graph_for_hybrid('full GraphTraversal demo')
    assert kg_facade is not None and kg_graph is not None
    traversal_svc = kg_facade.build_traversal(kg_graph)
    global kg_traversal
    kg_traversal = traversal_svc

    sid = _pick_demo_start_id(start_id)
    if not sid:
        raise RuntimeError('No start id_str available for traversal demo.')

    print('=== GraphTraversal modes demo (direct service) ===')
    print('start_id:', sid)
    print('max_depth:', max_depth)
    results: dict[str, TraversalResult] = {}
    for mode in modes:
        result = traversal_svc.traverse(sid, mode=mode, max_depth=max_depth)  # type: ignore[arg-type]
        results[mode] = result
        print(
            f'  mode={mode:10} visited={len(result.visited_ids):4} '
            f'edges={len(result.visited_edges):4} paths={len(result.paths):4}'
        )
    return results


def run_evidence_context_demo(
    start_id: str | None = None,
    *,
    mode: str = 'basis',
    max_depth: int = 2,
    filter_profile: str = 'current_law',
) -> EvidenceContext:
    """Build EvidenceContext + citation context via ContextBuilder (through facade)."""
    require_graph_for_hybrid('EvidenceContext demo')
    assert kg_facade is not None and kg_graph is not None

    sid = _pick_demo_start_id(start_id)
    if not sid:
        raise RuntimeError('No start id_str available for evidence context demo.')

    traversal = kg_facade.traverse(
        kg_graph,
        start_id=sid,
        mode=mode,  # type: ignore[arg-type]
        max_depth=max_depth,
    )
    overlays = document_overlays if graph_load_status.overlays_ready else {}
    constraints = QueryConstraints(validity_groups=('active', 'partial', 'future'))
    evidence = kg_facade.build_evidence_context(
        graph=kg_graph,
        traversal=traversal,
        overlays=overlays,
        filter_profile=filter_profile,  # type: ignore[arg-type]
        constraints=constraints,
    )
    citations = kg_facade.build_citation_context(
        graph=kg_graph,
        traversal=traversal,
        overlays=overlays,
        filter_profile=filter_profile,  # type: ignore[arg-type]
        constraints=constraints,
    )
    print('=== EvidenceContext / citation context demo ===')
    print('start_id:', sid, '| traversal_mode:', mode)
    print('filter empty_warning:', evidence.filter.empty_filter_warning)
    print('filter profile:', evidence.filter.filter_profile)
    print('documents in evidence:', len(evidence.documents))
    print('overlays attached:', len(evidence.overlays))
    print('paths:', len(evidence.paths))
    print('warnings:', evidence.warnings or ())
    print('citation ids (sample):', list(citations)[:8])
    return evidence


def run_overlay_joiner_sample(limit: int = 3) -> dict[str, Any]:
    """Show OverlayJoiner direct usage + currency histogram sample (no mutation)."""
    out: dict[str, Any] = {'ready': False}
    if kg_overlay_joiner is None:
        print('OverlayJoiner not wired (facade missing).')
        return out
    if not graph_load_status.overlays_ready or not document_overlays:
        print('Overlays unavailable — OverlayJoiner is imported/wired but no overlay files joined.')
        out['ready'] = False
        out['reason'] = 'overlays_missing'
        return out

    sample_ids = list(document_overlays.keys())[: max(1, limit)]
    samples = []
    for id_str in sample_ids:
        ov = document_overlays[id_str]
        samples.append({
            'id_str': id_str,
            'currency_status': ov.currency_status,
            'currency_status_as_of': ov.currency_status_as_of,
            'legal_authority_rank': ov.legal_authority_rank,
            'authority_rank_source': ov.authority_rank_source,
        })
    hist: dict[str, int] = {}
    for ov in document_overlays.values():
        hist[ov.currency_status] = hist.get(ov.currency_status, 0) + 1
    print('=== OverlayJoiner sample (joined DocumentOverlay) ===')
    print('joiner class:', type(kg_overlay_joiner).__name__)
    print('document_overlays:', len(document_overlays))
    print('currency histogram:', hist)
    print('sample rows:')
    for row in samples:
        print(' -', row)
    # utils smoke (same helpers overlay parsers use)
    print('utils smoke: as_int("3")=', as_int('3'), 'as_bool("yes")=', as_bool('yes'),
          'quality_flags([" a ", ""])=', quality_flags([' a ', '']))
    out.update({'ready': True, 'histogram': hist, 'samples': samples})
    return out


def run_loader_builder_optional() -> dict[str, Any]:
    """Optional JSONL parse/build visibility — only when rebuild is permitted.

    Colab-safe pickle sessions skip this to avoid multi-GB structural rebuilds.
    """
    report: dict[str, Any] = {'ran': False}
    if kg_facade is None:
        print('Facade missing — skip loader/builder optional demo.')
        return report
    if graph_load_status.graph_source_mode == 'jsonl_rebuild' and kg_build_result is not None:
        stats = kg_build_result.stats
        print('=== Loader/Builder already exercised via JSONL rebuild ===')
        print('documents:', stats.document_count, 'chunks:', stats.chunk_count,
              'verified_edges:', stats.verified_document_edge_count)
        report.update({'ran': True, 'path': 'jsonl_rebuild_already', 'stats': stats})
        return report
    if not ALLOW_JSONL_GRAPH_REBUILD:
        print(
            '=== Loader/Builder optional demo SKIPPED ===\n'
            '  Structural JSONL parse/build is heavy. Set ALLOW_JSONL_GRAPH_REBUILD=True\n'
            '  (and prefer unconstrained profile) to exercise GraphLoader.parse_sources / GraphBuilder.\n'
            f'  Current mode={graph_load_status.graph_source_mode}; pickle preferred under Colab-safe.'
        )
        report.update({'ran': False, 'reason': 'allow_jsonl_rebuild_false'})
        return report

    print('=== Optional facade.parse_sources (JSONL) ===')
    try:
        parsed = kg_facade.parse_sources()
        global kg_parsed_sources
        kg_parsed_sources = parsed
        print('ParsedGraphSources counts:')
        print('  documents:', len(parsed.documents))
        print('  external_stubs:', len(parsed.external_stubs))
        print('  provisions:', len(parsed.provisions))
        print('  chunks:', len(parsed.chunks))
        print('  edges:', len(parsed.edges))
        print('  text_provenance keys:', len(parsed.text_provenance))
        report.update({
            'ran': True,
            'path': 'parse_sources',
            'counts': {
                'documents': len(parsed.documents),
                'external_stubs': len(parsed.external_stubs),
                'provisions': len(parsed.provisions),
                'chunks': len(parsed.chunks),
                'edges': len(parsed.edges),
            },
        })
    except Exception as exc:
        print('parse_sources failed:', exc)
        report.update({'ran': False, 'error': str(exc)})
    return report


def run_full_graph_module_demo(
    *,
    start_id: str | None = None,
    include_jsonl_parse: bool | None = None,
) -> dict[str, Any]:
    """Orchestrate inventory + traversal + evidence + overlay + optional parse demos."""
    summary: dict[str, Any] = {
        'inventory': [],
        'traversal': None,
        'evidence': None,
        'overlay': None,
        'loader_builder': None,
    }
    summary['inventory'] = print_graph_module_inventory()
    if not graph_load_status.structural_ready or kg_graph is None or kg_facade is None:
        print('Structural graph not ready — full module demos that need a live graph are skipped.')
        print('Imports still cover the entire package surface (see previous cell).')
        return summary

    summary['traversal'] = run_traversal_modes_demo(start_id=start_id, max_depth=GRAPH_GUIDED_MAX_DEPTH)
    summary['evidence'] = run_evidence_context_demo(
        start_id=start_id,
        mode=GRAPH_GUIDED_TRAVERSAL_MODE,
        max_depth=GRAPH_GUIDED_MAX_DEPTH,
        filter_profile=FILTER_PROFILE,
    )
    summary['overlay'] = run_overlay_joiner_sample()
    do_jsonl = ALLOW_JSONL_GRAPH_REBUILD if include_jsonl_parse is None else include_jsonl_parse
    if do_jsonl:
        summary['loader_builder'] = run_loader_builder_optional()
    else:
        summary['loader_builder'] = run_loader_builder_optional()  # still prints skip reason
    print('\nFull Graph Module demo complete (primary hybrid path unchanged).')
    return summary


# Default: run lightweight full-module demos when graph is ready.
# Heavy JSONL parse remains gated by ALLOW_JSONL_GRAPH_REBUILD.
ENABLE_FULL_GRAPH_MODULE_DEMO = globals().get('ENABLE_FULL_GRAPH_MODULE_DEMO', True)

if ENABLE_FULL_GRAPH_MODULE_DEMO:
    full_graph_module_demo = run_full_graph_module_demo()
else:
    print('ENABLE_FULL_GRAPH_MODULE_DEMO=False — inventory/services still defined; demos not run.')
    print_graph_module_inventory()


## 4.1 Optional: export payload cache to CSV (Stage E — opt-in)

Heavy export — skipped under Colab-safe Run-all unless `RUN_PAYLOAD_CSV_EXPORT=True`.


In [ ]:
import csv


def export_payloads_to_csv(csv_path: Path | None = None, limit: int | None = 5000) -> Path:
    """Export the SQLite payloads table to CSV for manual inspection.

    Defaults to the first `limit` rows to keep the CSV small and fast to open.
    Pass limit=None to export the full table (this can be as large as payloads.jsonl).
    """
    csv_path = csv_path or (INDEX_DIR / 'payloads_export.csv')
    t0 = time.perf_counter()

    query = 'SELECT line_no, payload FROM payloads ORDER BY line_no'
    params: tuple = ()
    if limit is not None:
        query += ' LIMIT ?'
        params = (limit,)

    rows = []
    for line_no, payload_text in store._conn.execute(query, params):
        payload = json.loads(payload_text)
        rows.append({'line_no': line_no, **payload})

    # Payload schemas can vary slightly between chunks, so build the CSV header
    # from the union of keys seen across the exported rows.
    fieldnames: list[str] = []
    seen: set[str] = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)

    with csv_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

    print(f'Exported {len(rows):,} rows to {csv_path} in {time.perf_counter() - t0:.2f}s')
    return csv_path


# Stage E opt-in: skip under Colab-safe Run-all unless flag True
if RUN_PAYLOAD_CSV_EXPORT:
    export_payloads_to_csv(limit=PAYLOAD_CSV_EXPORT_LIMIT)
    _ = capture_memory_snapshot(note='after_payload_csv_export')
    print(_.format_line())
else:
    print('RUN_PAYLOAD_CSV_EXPORT=False — skipped payload CSV export (Colab-safe default).')
    print('Call export_payloads_to_csv(limit=PAYLOAD_CSV_EXPORT_LIMIT) manually if needed.')


## 4.2 Optional: download / export payload_cache.sqlite (Stage E — opt-in)

Heavy export — skipped under Colab-safe Run-all unless `RUN_PAYLOAD_CACHE_EXPORT=True`.


In [ ]:
def export_payload_cache_sqlite(dest_dir: Path | None = None) -> Path:
    """Copy payload_cache.sqlite out of INDEX_DIR and offer it for download.

    - In Google Colab: triggers a browser download via `google.colab.files.download`.
    - If Google Drive is mounted at /content/drive: also copies the file there.
    - Otherwise: copies the file to `dest_dir` (defaults to the project root) so it is
      easy to locate for a manual download/copy.
    """
    import shutil

    src_path = store.cache_path
    if not src_path.exists():
        raise FileNotFoundError(f'payload_cache.sqlite not found at {src_path}')

    t0 = time.perf_counter()
    print(f'Source: {src_path} ({src_path.stat().st_size / 1024 / 1024:.2f} MB)')

    try:
        from google.colab import files as colab_files  # type: ignore
        in_colab = True
    except ImportError:
        colab_files = None
        in_colab = False

    drive_root = Path('/content/drive')
    if in_colab and drive_root.exists():
        drive_dest = drive_root / 'MyDrive' / 'faiss_payload_cache' / src_path.name
        drive_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, drive_dest)
        print(f'Copied to Google Drive: {drive_dest} in {time.perf_counter() - t0:.2f}s')
        return drive_dest

    if in_colab and colab_files is not None:
        print('Triggering browser download via Colab...')
        colab_files.download(str(src_path))
        print(f'Download triggered in {time.perf_counter() - t0:.2f}s')
        return src_path

    dest_dir = dest_dir or PROJECT_ROOT
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / src_path.name
    if dest_path.resolve() != src_path.resolve():
        shutil.copy2(src_path, dest_path)
    print(f'Copied to {dest_path} in {time.perf_counter() - t0:.2f}s')
    return dest_path


# Stage E opt-in: skip under Colab-safe Run-all unless flag True
if RUN_PAYLOAD_CACHE_EXPORT:
    export_payload_cache_sqlite()
    _ = capture_memory_snapshot(note='after_payload_cache_export')
    print(_.format_line())
else:
    print('RUN_PAYLOAD_CACHE_EXPORT=False — skipped payload_cache.sqlite export (Colab-safe default).')
    print('Call export_payload_cache_sqlite() manually if needed.')


## 5. Retrieval helper

In [ ]:
def search(
    query: str,
    top_n: int = TOP_N,
    filter_profile: str = FILTER_PROFILE,
    score_threshold: float | None = SCORE_THRESHOLD,
    expand_units: bool | None = None,
    graph_guided_filter=None,
    use_hybrid_retriever: bool = False,
):
    """Run VectorRetriever and return (display_rows, RetrievalResult).

    filter_profile: 'current_law' | 'broad' | 'historical' | 'graph_guided' (via graph_guided_filter)

    Expansion labeling:
    - expand_units=True with graph_expansion wired → mechanism is graph expansion (module path)
    - expand_units=True without graph_expansion → local_expand_units (payload same-provision)
    Prefer the two-stage hybrid helper for seed vs expanded diagnostics.
    """
    active = hybrid_retriever if (use_hybrid_retriever and hybrid_retriever is not None) else retriever

    if filter_profile == 'graph_guided' and graph_guided_filter is None:
        print(
            'graph_guided requested without GraphGuidedFilter. '
            'Use the optional graph-guided pre-filter demo, or pass graph_guided_filter=... '
            'Falling back to broad (pure vector).'
        )
        filter_profile = 'broad'

    search_t0 = time.perf_counter()
    result = active.retrieve(
        query,
        top_n=top_n,
        filter_profile=filter_profile if graph_guided_filter is None else 'graph_guided',
        score_threshold=score_threshold,
        expand_units=LOCAL_EXPAND_UNITS if expand_units is None else expand_units,
        graph_guided_filter=graph_guided_filter,
    )
    print(f'Retrieval completed in {time.perf_counter() - search_t0:.2f}s')
    rows = []
    for rank, chunk in enumerate(result.chunks, start=1):
        rows.append({
            'rank': rank,
            'chunk_id': chunk.chunk_id,
            'id_str': chunk.id_str,
            'citation': chunk.citation_anchor or chunk.citation_label,
            'title': chunk.title,
            'unit_type': chunk.unit_type,
            'validity_group': chunk.validity_group,
            'parent_unit_id': chunk.parent_unit_id,
            'vector_score': round(chunk.vector_score, 4),
            'rerank_score': round(chunk.rerank_score, 4),
            'text': chunk.chunk_text[:700],
        })
    return rows, result


def show_results(rows):
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(json.dumps(row, ensure_ascii=False, indent=2))


## 5.1 Benchmark helper

In [ ]:
def benchmark_search(query: str, repeats: int = 3, filter_profile: str = FILTER_PROFILE):
    timings = []
    for i in range(repeats):
        t0 = time.perf_counter()
        rows, result = search(query, top_n=TOP_N, filter_profile=filter_profile)
        elapsed = time.perf_counter() - t0
        timings.append(elapsed)
        print(f'Run {i + 1}/{repeats}: {elapsed:.2f}s, returned={len(result.chunks)}, candidates={result.total_candidates}')
    avg = sum(timings) / len(timings)
    print(f'Average retrieval time over {repeats} runs: {avg:.2f}s')
    return timings


## 6. Run a query (Stage B vector smoke)

Sample pure vector retrieval. Results should be labeled `vector_only` when graph expansion is not used.


In [ ]:
query_t0 = time.perf_counter()
query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'

# Primary run under default FILTER_PROFILE (pure vector / non-graph)
rows, result = search(query, top_n=10, filter_profile=FILTER_PROFILE)
print('Filter profile used:', result.filter_profile_used)
print('Total candidates:', result.total_candidates)
print('Empty filter warning:', result.empty_filter_warning)
show_results(rows)
print(f'Query phase completed in {time.perf_counter() - query_t0:.2f}s')

# --- Filter-profile comparison (non-graph profiles remain fully usable without graph) ---
print('\n=== Filter profile comparison (same query) ===')
for profile in ('current_law', 'broad', 'historical'):
    _, r = search(query, top_n=TOP_N, filter_profile=profile)
    print(
        f'  {profile:12s} | candidates={r.total_candidates:4d} | '
        f'returned={len(r.chunks):2d} | empty_filter_warning={r.empty_filter_warning}'
    )

print(
    '  graph_guided  | secondary path — see optional demo cell '
    f'(ENABLE_GRAPH_GUIDED_PREFILTER_DEMO={ENABLE_GRAPH_GUIDED_PREFILTER_DEMO})'
)

# --- Local same-provision expansion demo (local_expand_units mechanism) ---
print('\n=== Local expansion demo (local_expand_units=True vs False) ===')
print('Mechanism label: local_expand_units (payload same-provision window; not graph_expansion)')
_, r_no = search(query, top_n=5, filter_profile='broad', expand_units=False)
_, r_yes = search(query, top_n=5, filter_profile='broad', expand_units=True)
print(f'  local_expand_units=False → {len(r_no.chunks)} chunks')
print(f'  local_expand_units=True  → {len(r_yes.chunks)} chunks')
parent_ids = {c.parent_unit_id for c in r_yes.chunks if c.parent_unit_id}
print(f'  unique parent_unit_id among expanded results: {len(parent_ids)}')

# Colab-safe: large multi-profile loops are opt-in Stage E
if not RUN_FILTER_PROFILE_COMPARISON:
    print('RUN_FILTER_PROFILE_COMPARISON=False — keep vector smoke lightweight under Colab-safe.')


## 6.1 Hybrid expansion demo (Stage C smoke)

Seed vs graph-expanded evidence, overlay signals, vector-only vs hybrid comparison, and optional graph-guided pre-filter (secondary, off under Colab-safe).

Labels: **graph_expansion** is distinct from **local_expand_units**. Hybrid success requires a loaded structural graph (preferably pickle).


In [ ]:
# ### HYBRID_GRAPH_INTEGRATION — diagnostics, comparison, demos (US2/US3/US4)

hybrid_demo_query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'


def show_hybrid_diagnostics(ctx: HybridEvidenceContext) -> None:
    """Seed vs expanded counts, identity samples, overlays, warnings (US2)."""
    print('=== Hybrid diagnostics ===')
    print('Query:', ctx.query)
    print('Mode:', ctx.mode)
    print('Mechanism: graph_expansion (not local_expand_units)')
    print(f'Seed count: {len(ctx.seed.seed_chunks)} | seed candidates: {ctx.seed.total_candidates}')
    if ctx.expansion is None:
        print('Expansion: not run')
    else:
        print(
            f'Expanded ids: {len(ctx.expansion.expanded_chunk_ids)} | '
            f'added: {len(ctx.expansion.added_chunk_ids)} | '
            f'capped: {ctx.expansion.capped} | '
            f'mechanism_label: {ctx.expansion.mechanism_label}'
        )
        if ctx.expansion.warnings:
            print('Expansion warnings:')
            for w in ctx.expansion.warnings:
                print(' -', w)
    print(f'Evidence (citation-ready) count: {len(ctx.evidence_chunks)}')
    print('Sample identity chain (chunk_id → parent_unit_id → id_str):')
    show_results(chunks_to_display_rows(ctx.evidence_chunks, limit=8))

    print('\n--- Overlay diagnostics ---')
    if ctx.overlay_available and ctx.document_overlays:
        print(f'Overlay coverage for involved docs: {len(ctx.document_overlays)}')
        sample_id, sample_ov = next(iter(ctx.document_overlays.items()))
        print(f'Sample document id_str={sample_id}')
        print(f'  currency_status: {sample_ov.currency_status}')
        print(f'  currency_status_as_of: {sample_ov.currency_status_as_of}')
        print(f'  legal_authority_rank: {sample_ov.legal_authority_rank}')
        print(f'  authority_rank_source: {sample_ov.authority_rank_source}')
    else:
        print('Overlays unavailable or none matched involved documents — no authoritative currency/authority claims.')

    if ctx.diagnostics:
        print('\nStage notes:')
        for note in ctx.diagnostics:
            print(' -', note)


def compare_vector_vs_hybrid(
    query: str,
    *,
    top_n: int = TOP_N,
    filter_profile: str = FILTER_PROFILE,
) -> ModeComparisonRecord:
    """Same query under vector_only vs hybrid_expanded (US3 / FR-011)."""
    notes: list[str] = []
    # Vector-only
    vo_result = retriever.retrieve(
        query,
        top_n=top_n,
        filter_profile=filter_profile,
        score_threshold=SCORE_THRESHOLD,
        expand_units=False,
    )
    vo_ids = [c.chunk_id for c in vo_result.chunks]

    expansion_ran = False
    added = 0
    hybrid_ids: list[str] = []
    hybrid_count = 0
    try:
        require_graph_for_hybrid('hybrid side of mode comparison')
        hctx = run_hybrid_retrieve(query, top_n=top_n, filter_profile=filter_profile, enable_expansion=True)
        expansion_ran = hctx.expansion is not None and hctx.mode == 'hybrid_expanded'
        added = len(hctx.expansion.added_chunk_ids) if hctx.expansion else 0
        hybrid_ids = [c.chunk_id for c in hctx.evidence_chunks]
        hybrid_count = len(hctx.evidence_chunks)
        if expansion_ran and added == 0:
            notes.append('Expansion ran but added nothing beyond seeds.')
        elif expansion_ran:
            notes.append(f'Expansion added {added} chunk ids.')
        notes.extend(hctx.diagnostics[:5])
    except RuntimeError as exc:
        notes.append(f'Hybrid unavailable: {exc}')
        hybrid_count = -1

    record = ModeComparisonRecord(
        query=query,
        vector_only_count=len(vo_result.chunks),
        hybrid_count=hybrid_count,
        expansion_ran=expansion_ran,
        added_context_count=added,
        sample_vector_only_ids=vo_ids[:5],
        sample_hybrid_ids=hybrid_ids[:5],
        notes=notes,
    )
    print('=== Mode comparison: vector_only vs hybrid_expanded ===')
    print('Query:', query)
    print(f'  vector_only     count={record.vector_only_count} sample_ids={record.sample_vector_only_ids}')
    print(f'  hybrid_expanded count={record.hybrid_count} expansion_ran={record.expansion_ran} added={record.added_context_count}')
    print(f'  sample_hybrid_ids={record.sample_hybrid_ids}')
    for n in record.notes:
        print('  note:', n)
    return record


def run_graph_guided_prefilter_demo(
    query: str,
    *,
    start_id: str | None = None,
    top_n: int = TOP_N,
    filter_profile: str = 'current_law',
) -> GraphGuidedDemoResult:
    """Secondary whitelist-before-search path (US4 / FR-020). Not the default ask() path."""
    require_graph_for_hybrid('graph-guided pre-filter demo')
    assert kg_facade is not None and kg_graph is not None

    sid = (start_id or GRAPH_GUIDED_START_ID or '').strip()
    if not sid:
        # Derive from a seed hit when possible
        seed = retriever.retrieve(query, top_n=max(1, top_n), filter_profile=FILTER_PROFILE, expand_units=False)
        if seed.chunks and seed.chunks[0].id_str:
            sid = seed.chunks[0].id_str
            print(f'Graph-guided start id_str taken from first seed hit: {sid}')
        else:
            # Fall back to first document with a verified edge
            for edge in kg_graph.verified_document_edges:
                if edge.src_id in kg_graph.documents:
                    sid = edge.src_id
                    print(f'Graph-guided start id_str taken from verified edge src: {sid}')
                    break
    if not sid:
        raise RuntimeError('No start id_str available for graph-guided pre-filter demo.')

    traversal = kg_facade.traverse(
        kg_graph,
        start_id=sid,
        mode=GRAPH_GUIDED_TRAVERSAL_MODE,  # type: ignore[arg-type]
        max_depth=GRAPH_GUIDED_MAX_DEPTH,
    )
    overlays = document_overlays if graph_load_status.overlays_ready else {}
    guided = kg_facade.build_graph_guided_filter(
        graph=kg_graph,
        traversal=traversal,
        overlays=overlays,
        filter_profile=filter_profile,  # type: ignore[arg-type]
        constraints=QueryConstraints(validity_groups=('active', 'partial', 'future')),
    )
    print('=== Graph-guided pre-filter demo (SECONDARY path) ===')
    print('start_id:', sid)
    print('traversal_mode:', GRAPH_GUIDED_TRAVERSAL_MODE)
    print('whitelist size:', len(guided.id_strs))
    print('empty_filter_warning:', guided.empty_filter_warning)
    print('filter_profile:', guided.filter_profile)
    print('filter reason:', getattr(guided, 'reason', '') or '(none)')

    retrieval = None
    if guided.empty_filter_warning or not guided.id_strs:
        print(
            'EMPTY whitelist — not searching full corpus under a graph-guided label. '
            'empty_filter_warning stays True; no unfiltered hits returned as graph-guided.'
        )
        retrieval = RetrievalResult([], 0, 'graph_guided', empty_filter_warning=True)
    else:
        retrieval = retriever.retrieve(
            query,
            top_n=top_n,
            graph_guided_filter=guided,
            expand_units=False,
        )
        print('graph-guided retrieval returned:', len(retrieval.chunks), 'chunks')
        print('empty_filter_warning on result:', retrieval.empty_filter_warning)
        show_results(chunks_to_display_rows(retrieval.chunks, limit=8))

    return GraphGuidedDemoResult(
        start_id=sid,
        traversal_mode=str(GRAPH_GUIDED_TRAVERSAL_MODE),
        whitelist_size=len(guided.id_strs),
        empty_filter_warning=bool(guided.empty_filter_warning or not guided.id_strs),
        filter_reason=str(getattr(guided, 'reason', '') or guided.filter_profile),
        retrieval=retrieval,
    )


# --- Demo runs (safe when graph missing: hybrid calls fail clearly) ---
print('Running hybrid demo query when enabled...')
hybrid_ctx = None
comparison_record = None
graph_guided_demo = None

if ENABLE_HYBRID_EXPANSION and graph_load_status.structural_ready:
    hybrid_ctx = run_hybrid_retrieve(hybrid_demo_query, top_n=TOP_N, filter_profile=FILTER_PROFILE)
    show_hybrid_diagnostics(hybrid_ctx)
    comparison_record = compare_vector_vs_hybrid(hybrid_demo_query)
elif ENABLE_HYBRID_EXPANSION and not graph_load_status.structural_ready:
    print('Hybrid enabled but graph unavailable — demonstrating FR-015 clear failure:')
    try:
        require_graph_for_hybrid('hybrid demo')
    except RuntimeError as exc:
        print('Expected failure:', exc)
    print('Pure vector search still works:')
    rows_v, res_v = search(hybrid_demo_query, top_n=5, filter_profile=FILTER_PROFILE, expand_units=False)
    print('vector_only returned:', len(res_v.chunks))
else:
    print('ENABLE_HYBRID_EXPANSION=False — skip hybrid demo. Vector-only path remains default.')

if ENABLE_GRAPH_GUIDED_PREFILTER_DEMO:
    if graph_load_status.structural_ready:
        graph_guided_demo = run_graph_guided_prefilter_demo(hybrid_demo_query)
    else:
        print('Graph-guided demo enabled but graph unavailable — skipping with explicit message.')
else:
    print('ENABLE_GRAPH_GUIDED_PREFILTER_DEMO=False — secondary pre-filter demo not run.')


# --- Session outcome labels (FR-023) after hybrid demo ---
if hybrid_ctx is not None and getattr(hybrid_ctx, 'mode', None) == 'hybrid_expanded':
    print(
        format_session_outcome(
            session_outcome_label(
                colab_safe=COLAB_SAFE,
                structural_ready=graph_load_status.structural_ready,
                loaded_from_pickle=graph_load_status.loaded_from_pickle,
                hybrid_used=True,
                vector_ok=True,
            )
        )
    )
elif ENABLE_HYBRID_EXPANSION and not graph_load_status.structural_ready:
    print(
        format_session_outcome(
            'hybrid_unavailable',
            detail='Hybrid demo requested graph; vector_only path still OK.',
        )
    )
else:
    print(
        format_session_outcome(
            session_outcome_label(
                colab_safe=COLAB_SAFE,
                structural_ready=graph_load_status.structural_ready,
                loaded_from_pickle=getattr(graph_load_status, 'loaded_from_pickle', False),
                hybrid_used=False,
                vector_ok=True,
            )
        )
    )

# FULL_GRAPH_MODULE demos run in the dedicated Stage C cell (§4.4),
# not inside this hybrid smoke cell — keeps primary hybrid path focused.
if 'print_graph_module_inventory' in globals() and graph_load_status.structural_ready:
    print('Full Graph Module helpers available: inventory_graph_modules, '
          'run_traversal_modes_demo, run_evidence_context_demo, run_full_graph_module_demo')


## 7. Optional: inspect one full chunk

In [ ]:
if result.chunks:
    chunk = result.chunks[0]
    print('chunk_id:', chunk.chunk_id)
    print('citation:', chunk.citation_anchor or chunk.citation_label)
    print('title:', chunk.title)
    print('scores:', {'vector': chunk.vector_score, 'rerank': chunk.rerank_score})
    print('--- text ---')
    print(chunk.chunk_text)
    print('--- metadata keys ---')
    print(sorted(chunk.metadata.keys()))


## 7.1 Optional cleanup between stages (best-effort)

Drop export frames or unload the structural graph to free headroom before generation or after heavy demos. Not an OS hard memory reservation.

If the graph is unloaded, hybrid becomes unavailable until Stage C reloads.


In [ ]:
# ### COLAB_SAFE_RAM_FIT — best-effort cleanup / release helper (FR-012)


def release_optional_objects(
    *,
    drop_export_frames: bool = True,
    drop_comparison_records: bool = True,
    unload_overlays: bool = False,
    unload_structural_graph: bool = False,
    run_gc: bool = True,
) -> list[str]:
    """Drop optional heavy notebook objects. Not an OS memory reservation.

    If unload_structural_graph=True, hybrid becomes unavailable until Stage C reloads.
    """
    req = CleanupRequest(
        drop_export_frames=drop_export_frames,
        drop_comparison_records=drop_comparison_records,
        unload_overlays=unload_overlays,
        unload_structural_graph=unload_structural_graph,
        run_gc=run_gc,
    )
    actions = apply_cleanup(globals(), req)
    print('Cleanup actions:')
    for a in actions:
        print(' -', a)
    print(
        format_resident_snapshot(
            ResidentComponentSnapshot(
                store_loaded='store' in globals() and store is not None,
                embedder_loaded='embedder' in globals() and embedder is not None,
                structural_graph_loaded=bool(
                    globals().get('graph_load_status')
                    and graph_load_status.structural_ready
                    and globals().get('kg_graph') is not None
                ),
                graph_source_mode=getattr(globals().get('graph_load_status'), 'graph_source_mode', None),
                overlays_loaded=bool(getattr(globals().get('graph_load_status'), 'overlays_ready', False)),
                hybrid_retriever_ready=globals().get('hybrid_retriever') is not None,
                generator_configured=bool(globals().get('generator')),
                optional_frames_held=[],
            )
        )
    )
    return actions


print('release_optional_objects() defined. Example:')
print('  release_optional_objects()  # drop export/comparison frames + gc')
print('  release_optional_objects(unload_structural_graph=True)  # hybrid unavailable until reload')


## 8. Configure the answer generator (Stage D — optional)

Set `LLM_BASE_URL`, `LLM_API_KEY`, and `LLM_MODEL_NAME` via environment variables so credentials never end up hardcoded in this notebook. Any OpenAI-compatible chat completions endpoint works (OpenAI, Azure OpenAI, vLLM, Together, OpenRouter, etc.).

```bash
export LLM_BASE_URL="https://api.your-provider.com/v1"
export LLM_API_KEY="..."
export LLM_MODEL_NAME="gpt-4o-mini"
```

If these are not set, the notebook still runs in retrieval/hybrid mode; generation is skipped with an explicit reason. Local in-process LLMs are out of scope for Colab-safe RAM guarantees.


In [ ]:
from generation.reasoning_client import GeneratorConfig

generator_config = GeneratorConfig(
    base_url=os.environ.get('LLM_BASE_URL', '').strip(),
    api_key=os.environ.get('LLM_API_KEY', '').strip(),
    model_name=os.environ.get('LLM_MODEL_NAME', '').strip(),
)

# Back-compat aliases used by older cells / mental model
BASE_URL = generator_config.base_url
API_KEY = generator_config.api_key
MODEL_NAME = generator_config.model_name

if not generator_config.is_complete():
    print('Generator not fully configured. Set LLM_BASE_URL, LLM_API_KEY, LLM_MODEL_NAME env vars to enable answer generation.')
else:
    print('Generator configured:')
    print('  BASE_URL:', BASE_URL)
    print('  MODEL_NAME:', MODEL_NAME)
    print('  API_KEY:', generator_config.masked_key())


## 9. Generator client and answer-generation helper

Uses the extracted module `generation.reasoning_client`:
- `GeneratorClient.generate` → `RawGenerationResponse`
- `parse_generation_response` — three shapes: dedicated reasoning field, `<think>...</think>` block, or `not_returned`
- `generate_answer` → `GenerationOutcome` (skip empty context / error / parsed)


In [ ]:
from generation.reasoning_client import (
    ANSWER_PROMPT,
    GenerationOutcome,
    GeneratorClient,
    ParsedAnswer,
    RawGenerationResponse,
    format_context_for_prompt,
    generate_answer as _generate_answer_outcome,
    parse_generation_response,
)

generator: GeneratorClient | None = None
if generator_config.is_complete():
    generator = GeneratorClient(
        base_url=generator_config.base_url,
        api_key=generator_config.api_key,
        model=generator_config.model_name,
    )
    print('Generator client ready.')
    print('ANSWER_PROMPT includes reasoning instruction:', 'reasoning' in ANSWER_PROMPT.lower() or 'suy luận' in ANSWER_PROMPT.lower())
else:
    print('Generator client not created (missing config). Retrieval-only mode.')


def generate_answer(query: str, chunks, *, qa_id: str | None = None) -> GenerationOutcome:
    """Notebook wrapper: returns GenerationOutcome (never raises for empty context)."""
    gen_t0 = time.perf_counter()
    outcome = _generate_answer_outcome(generator, query, chunks, qa_id=qa_id)
    print(f'Generation completed in {time.perf_counter() - gen_t0:.2f}s')
    return outcome


def display_generation_outcome(outcome: GenerationOutcome) -> None:
    """Print answer and reasoning as two distinct sections (FR-020/FR-021)."""
    if outcome.skipped_empty_context:
        print('Skipped generation: empty retrieved context.')
        return
    if outcome.error:
        print('--- Generation error ---')
        print(outcome.error)
        return
    parsed = outcome.parsed
    assert parsed is not None
    print('\n--- Answer ---')
    print(parsed.answer or '(empty)')
    print('\n--- Reasoning ---')
    if parsed.reasoning_available and parsed.reasoning:
        print(f'(source={parsed.reasoning_source})')
        print(parsed.reasoning)
    else:
        print('not returned by this model')


## 10. Full RAG pipeline: retrieve + generate (Stage D)

Ad hoc `ask()` runs retrieval then generation. Final answer and model reasoning are shown as **two distinct sections**. If the model does not return reasoning, the notebook prints `not returned by this model` rather than inventing text.

Hybrid `ask()` requires a loaded structural graph; pure vector `ask(..., use_hybrid=False)` works after Stage B alone.


In [ ]:
def ask(
    query: str,
    top_n: int = TOP_N,
    filter_profile: str = FILTER_PROFILE,
    score_threshold: float | None = SCORE_THRESHOLD,
    *,
    use_hybrid: bool | None = None,
    use_hybrid_evidence: bool | None = None,
):
    """Full pipeline: vector seed → (optional graph expand/overlays) → generate.

    Default demonstration path uses hybrid expanded evidence when
    ENABLE_HYBRID_EXPANSION and the graph is loaded (FR-009 / FR-022).
    Hybrid requested while graph unavailable fails clearly (FR-015).
    """
    hybrid = ENABLE_HYBRID_EXPANSION if use_hybrid is None else use_hybrid
    hybrid_for_gen = USE_HYBRID_EVIDENCE_FOR_GENERATION if use_hybrid_evidence is None else use_hybrid_evidence

    hybrid_ctx = None
    result = None
    evidence_chunks = []

    if hybrid:
        require_graph_for_hybrid('hybrid ask() full pipeline')
        hybrid_ctx = run_hybrid_retrieve(
            query,
            top_n=top_n,
            filter_profile=filter_profile,
            score_threshold=score_threshold,
            enable_expansion=True,
        )
        print('Mode:', hybrid_ctx.mode)
        print('Seed candidates:', hybrid_ctx.seed.total_candidates)
        print('Evidence chunks:', len(hybrid_ctx.evidence_chunks))
        print('Expansion added context:', hybrid_ctx.expansion_added_context)
        if hybrid_ctx.diagnostics:
            print('Diagnostics:')
            for note in hybrid_ctx.diagnostics:
                print(' -', note)
        show_results(chunks_to_display_rows(hybrid_ctx.evidence_chunks))
        evidence_chunks = list(hybrid_ctx.evidence_chunks)
        # Build a RetrievalResult-like object for callers that expect .chunks
        from retrieval.schema import RetrievalResult
        result = RetrievalResult(
            chunks=evidence_chunks,
            total_candidates=hybrid_ctx.seed.total_candidates,
            filter_profile_used=filter_profile,
            empty_filter_warning=False,
        )
    else:
        rows, result = search(
            query,
            top_n=top_n,
            filter_profile=filter_profile,
            score_threshold=score_threshold,
            expand_units=False,
        )
        print('Mode: vector_only')
        print('Filter profile used:', result.filter_profile_used)
        print('Total candidates:', result.total_candidates)
        print('Empty filter warning:', result.empty_filter_warning)
        show_results(rows)
        evidence_chunks = list(result.chunks)

    usable = [c for c in evidence_chunks if (c.chunk_text or '').strip()]
    if not usable:
        print('No usable evidence text after retrieval/expansion; skipping generation (empty context).')
        outcome = GenerationOutcome(qa_id=None, parsed=None, skipped_empty_context=True, error=None)
        return {
            'query': query,
            'outcome': outcome,
            'result': result,
            'hybrid': hybrid_ctx,
            'mode': hybrid_ctx.mode if hybrid_ctx else 'vector_only',
        }

    if generator is None:
        print('Generator not configured; returning retrieval/expansion-only result.')
        return {
            'query': query,
            'outcome': None,
            'result': result,
            'hybrid': hybrid_ctx,
            'mode': hybrid_ctx.mode if hybrid_ctx else 'vector_only',
        }

    if hybrid and hybrid_for_gen:
        gen_chunks = usable
        print('Generation uses hybrid expanded evidence (USE_HYBRID_EVIDENCE_FOR_GENERATION=True).')
    elif hybrid and not hybrid_for_gen:
        # Prefer seed-only evidence when hybrid retrieval ran but generation should not use expansion.
        seed_only = list(hybrid_ctx.seed.seed_chunks) if hybrid_ctx is not None else usable
        gen_chunks = [c for c in seed_only if (c.chunk_text or '').strip()] or usable
        print('Generation uses seed-only evidence (USE_HYBRID_EVIDENCE_FOR_GENERATION=False).')
    else:
        gen_chunks = usable
    outcome = generate_answer(query, gen_chunks)
    display_generation_outcome(outcome)

    print('\n--- Citations used (citation-ready evidence only) ---')
    for rank, chunk in enumerate(gen_chunks, start=1):
        print(
            f'[{rank}] {chunk.citation_anchor or chunk.citation_label} - {chunk.title} '
            f'| chunk_id={chunk.chunk_id} → parent_unit_id={chunk.parent_unit_id} → id_str={chunk.id_str}'
        )
    return {
        'query': query,
        'outcome': outcome,
        'result': result,
        'hybrid': hybrid_ctx,
        'mode': hybrid_ctx.mode if hybrid_ctx else 'vector_only',
    }


pipeline_query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'
# Uncomment when ready (generator optional — hybrid retrieval still completes without credentials):
# pipeline_output = ask(pipeline_query, top_n=10, filter_profile='broad')
print('ask() defined. Default path: vector seed → graph expand/overlays → generate when hybrid enabled.')
print('Call: pipeline_output = ask(pipeline_query, top_n=10, filter_profile="broad")')
print('Vector-only: pipeline_output = ask(pipeline_query, use_hybrid=False)')


## 11. Optional benchmark sample (Stage E — opt-in)

`run_benchmark_sample` scores retrieval hit-rate against `data/benchmark/qa_final.jsonl` and, when a generator is configured, records per-question `GenerationOutcome`. Auto-run is gated by `RUN_BENCHMARK_SAMPLE` (default **False** under Colab-safe).

This is a demo/validation path — not a full LLM-as-judge evaluation. Use `scripts/evaluate_e2e.py` for judged scoring.


In [ ]:
import random


def run_benchmark_sample(
    qa_path: Path | None = None,
    sample_size: int = BENCHMARK_SAMPLE_SIZE,
    filter_profile: str = FILTER_PROFILE,
    seed: int = 42,
    run_generation: bool = True,
):
    """Run retrieval (+ generation, if configured) over a random sample of qa_final.jsonl.

    Reports per-question retrieval hit (whether any ground-truth chunk/provision/document id
    appears among the retrieved results) plus an aggregate hit rate and average latency.
    Unanswerable questions (empty ground_truth) are excluded from the hit-rate denominator.

    Generation reuses already-retrieved chunks (no re-query). Failures are recorded per
    question and do not abort the loop.
    """
    qa_path = qa_path or (PROJECT_ROOT / 'data' / 'benchmark' / 'qa_final.jsonl')
    if not qa_path.exists():
        raise FileNotFoundError(f'Benchmark file not found at {qa_path}')

    with qa_path.open('r', encoding='utf-8') as f:
        all_cases = [json.loads(line) for line in f if line.strip()]

    rng = random.Random(seed)
    sample = rng.sample(all_cases, min(sample_size, len(all_cases)))

    records = []
    latencies = []
    hits = 0
    scored = 0
    gen_errors = 0
    gen_skipped = 0
    gen_ok = 0

    for qa in sample:
        question = qa.get('question') or ''
        ground_truth = qa.get('ground_truth') or {}
        gt_ids = (
            set(ground_truth.get('chunk_ids') or [])
            | set(ground_truth.get('provision_ids') or [])
            | set(ground_truth.get('document_ids') or [])
        )
        is_unanswerable = qa.get('answer_type') == 'unanswerable' or not gt_ids

        t0 = time.perf_counter()
        _, result = search(question, top_n=TOP_N, filter_profile=filter_profile)
        elapsed = time.perf_counter() - t0
        latencies.append(elapsed)

        retrieved_ids = set()
        for chunk in result.chunks:
            retrieved_ids.update({chunk.chunk_id, chunk.parent_unit_id, chunk.id_str})

        hit = bool(gt_ids & retrieved_ids)
        if not is_unanswerable:
            scored += 1
            if hit:
                hits += 1

        outcome: GenerationOutcome | None = None
        if run_generation and generator is not None:
            # generate_answer never raises for empty context; API errors become outcome.error
            outcome = _generate_answer_outcome(
                generator,
                question,
                result.chunks,
                qa_id=qa.get('qa_id'),
            )
            if outcome.skipped_empty_context:
                gen_skipped += 1
            elif outcome.error:
                gen_errors += 1
            elif outcome.parsed is not None:
                gen_ok += 1

        parsed = outcome.parsed if outcome else None
        records.append({
            'qa_id': qa.get('qa_id'),
            'question': question,
            'answer_type': qa.get('answer_type'),
            'category': qa.get('category'),
            'is_unanswerable': is_unanswerable,
            'retrieval_hit': hit,
            'total_candidates': result.total_candidates,
            'latency_s': round(elapsed, 3),
            'generated_answer': parsed.answer if parsed else None,
            'reasoning': parsed.reasoning if parsed else None,
            'reasoning_source': parsed.reasoning_source if parsed else None,
            'reasoning_available': parsed.reasoning_available if parsed else False,
            'generation_error': outcome.error if outcome else None,
            'skipped_empty_context': outcome.skipped_empty_context if outcome else False,
        })

    hit_rate = hits / scored if scored else float('nan')
    avg_latency = sum(latencies) / len(latencies) if latencies else float('nan')

    print(f'Sampled {len(sample)} questions ({scored} scored, {len(sample) - scored} unanswerable excluded)')
    print(f'Hit rate: {hit_rate:.2%}' if scored else 'Hit rate: n/a (no scored questions)')
    print(f'Average retrieval latency: {avg_latency:.3f}s')
    if run_generation and generator is not None:
        print(f'Generation: ok={gen_ok}, skipped_empty={gen_skipped}, errors={gen_errors}')

    return {
        'records': records,
        'hit_rate': hit_rate,
        'avg_latency_s': avg_latency,
        'sample_size': len(sample),
        'scored': scored,
        'generation_ok': gen_ok,
        'generation_errors': gen_errors,
        'generation_skipped': gen_skipped,
    }


# Stage E: never auto-run under Colab-safe unless RUN_BENCHMARK_SAMPLE=True
if RUN_BENCHMARK_SAMPLE:
    benchmark_summary = run_benchmark_sample(
        sample_size=BENCHMARK_SAMPLE_SIZE, run_generation=False
    )
else:
    print(
        'RUN_BENCHMARK_SAMPLE=False — skipped auto benchmark '
        f'(Colab-safe default). Function ready; size would be {BENCHMARK_SAMPLE_SIZE}.'
    )
    print('Example: benchmark_summary = run_benchmark_sample(sample_size=BENCHMARK_SAMPLE_SIZE)')
